In [27]:
!pip install transliterate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
from typing import Tuple, List, Optional
import string
import re
import logging
import argparse
import os
from collections import defaultdict

from transliterate import translit
from tqdm import tqdm
import transformers
from transformers import AutoTokenizer, AutoModel
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
# import pyarrow as pa
# import pyarrow.parquet as pq




In [29]:
LATIN_CHARS = set(string.ascii_lowercase)
DIGITS = set(string.digits)
SPECIAL_CHARS = set('.-_')
CYRILLIC_CHARS = set('абвгдежзийклмнопрстуфхцчшщъыьэюя')  # кроме ё, потому что не встречается

POSSIBLE_CHARS = set.union(LATIN_CHARS, DIGITS, SPECIAL_CHARS, CYRILLIC_CHARS)


In [30]:

# logging.basicConfig(level=logging.INFO)
# logger = logging.getLogger(__name__)



In [31]:

import logging
import sys

# Create a logger object
logger = logging.getLogger(__name__)

# Remove any existing handlers associated with the logger object (useful in Jupyter)
if logger.hasHandlers():
    logger.handlers.clear()

# Set the logger level to INFO
logger.setLevel(logging.INFO)

# Add StreamHandler to print to the console
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

# Add a simple formatter
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(console_handler)


In [32]:
class UrlTokenizer:
    def __init__(self, tokenizer, preprocessing_needed: bool) -> None:
        self.tokenizer = tokenizer
        self.preprocessing_needed = preprocessing_needed
    
    def _get_words(self, url) -> Tuple[str, List[str]]:
        return get_words(url, self.preprocessing_needed)
    
    def tokenize_with_status(self, url: str) -> Tuple[str, torch.Tensor]:
        status, words = self._get_words(url)
        words_str = ' '.join(words)
        tokenized = self.tokenizer(words_str, return_tensors="pt")
        return status, tokenized
    
    def tokenize(self, url: str) -> torch.Tensor:
        return self.tokenize_with_status(url)[1]
    
    def __call__(self, url: str) -> torch.Tensor:
        return self.tokenize(url)


#############  Seq Embedders  #############

class SeqEmbedderBase(torch.nn.Module):
    def __init__(self, model) -> None:
        super().__init__()
        self.model = model
    
    def embed(self, tokenized: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError
    
    def __call__(self, tokenized: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.embed(tokenized, attention_mask)
    

class MeanPoolSeqEmbedder(SeqEmbedderBase):
    def embed(self, tokenized: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        model_output = self.model(tokenized, attention_mask)
        return mean_pooling(model_output, attention_mask)
    

class EmbedderPoolerOutput(SeqEmbedderBase):
    def embed(self, tokenized: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        model_output = self.model(tokenized, attention_mask)
        return model_output['pooler_output']


class EmbedderCLS(SeqEmbedderBase):
    def embed(self, tokenized: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        model_output = self.model(tokenized, attention_mask)
        cls_pos = 0
        return model_output['last_hidden_state'][:, cls_pos, :]


############ Data Loading utils ############

class UrlDataset(Dataset):
    def __init__(self, urls: List[str], tokenizer, preprocessing_needed: bool) -> None:
        self.urls = urls
        self.tokenizer = UrlTokenizer(tokenizer, preprocessing_needed)
    
    def __len__(self) -> int:
        return len(self.urls)
    
    def __getitem__(self, idx: int) -> torch.Tensor:
        url = self.urls[idx]
        input_ids = self.tokenizer(url)['input_ids'].squeeze(0)  # Squeeze to remove batch dimension
        mask = self.tokenizer(url)['attention_mask'].squeeze(0)
        return input_ids, mask

def collate_fn(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    
    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)  # Padding with 0
    padded_attention_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)  # Padding with 0
    
    return padded_input_ids, padded_attention_masks


############################
 
def is_punycode(s: str) -> bool:
    return s.startswith('xn--')    
    

def convert_punycode(puny_domain: str) -> str:
    """
    Converts punycode to unicode

    Example:
    >>> convert_punycode('xn--22-glcqfm3bya1b.xn--p1ai')
    <<< 'грузчик22.рф'
    """
    return puny_domain.encode().decode('idna')


def remove_extension(url: str) -> str:
    # I removed `ru-an` specifically, because it was the only
    # subdomain present in mixed cyrillic-latin domains. 
    # Without `ru-an` there are no mixed domains like коронавирус.ru-an.info
    
    #  Нужно ли удалять:
    # livejournal.com
    # turbopages.org
    
    url = ('.').join(url.split('.')[:-1])
    url = url[:-6] if url.endswith('ru-an') else url
    return url


def get_mixed_lang_domains(url_hosts: list) -> set:
    # To get knowledge about urls that contain 
    # both cyrillic and latin characters.
    mixed_lang_domains = set()

    for url in url_hosts:
        if is_punycode(url):
            url = convert_punycode(url)
        url = remove_extension(url)
        url_chars_set = set(url)
        
        if url_chars_set - LATIN_CHARS != url_chars_set and url_chars_set - CYRILLIC_CHARS != url_chars_set:
            # url has both cyrillic and latin
            mixed_lang_domains.add(url)
    return mixed_lang_domains


def convert_to_cyrillic(url: str) -> Tuple[str, str]:
    statuses = []
    url_set = set(url)
    has_latin =  (url_set - LATIN_CHARS) != url_set
    has_cyrillic =  (url_set - CYRILLIC_CHARS) != url_set 
    if has_latin:
#         assert not is_cyrillic, f"Встречен домен содержащий и латиницу и кириллицу: {url}"
        url = translit(url, 'ru')
    
        if has_cyrillic:
            statuses.append("transliterated. cyr_and_latin_before_translit")
            
        url_set = set(url)
        
        if ((url_set - CYRILLIC_CHARS) == url_set):
            statuses.append("no_cyrillic_after_translit")
        has_xwq = url_set.union({'x'}) == url_set or url_set.union({'w'}) == url_set or url_set.union({'q'}) == url_set
        if has_xwq:
            statuses.append("has_xwq")
        elif (url_set - LATIN_CHARS) != (url_set):
            statuses.append("unexpected_eng_left")
        else:
            statuses.append("proper_translit")
    elif has_cyrillic:
        # `has_crillic and has_latin` scenario is already processed.
        statuses.append('initially_ru')
    else:
        statuses.append("no_eng_no_ru")
    status = ' '.join(statuses)
    return status, url


def preprocess_url(url: str) -> str:
    if is_punycode(url):
        url = convert_punycode(url)
    url = remove_extension(url)
    return url

def get_words(url: str, preprocess: bool) -> Tuple[str, List[str]]:
    if preprocess:
        url = preprocess_url(url)
    status, url = convert_to_cyrillic(url)
    words = re.findall(r'[а-я]+', url)
    return status, words


# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    # torch.equal(model_output['last_hidden_state'], model_output[0]) returns true
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-10)
    return sum_embeddings / sum_mask


def get_output_file_name(input_file_path: str, model_name: str, extension: str) -> str:
    input_file_name = os.path.basename(input_file_path)
    model_name = model_name.replace('/', '_')
    return f'{input_file_name}_{model_name}.{extension}'


def save(data: Tuple[List[str], torch.Tensor], output_file_path: str):
    urls, embeddings = data
    embeddings_list = embeddings.tolist()
    data = {'url_host': urls, 'url_host_embedding': embeddings_list}
    df = pd.DataFrame(data)
    df.to_parquet(output_file_path, engine='pyarrow', index=False)
    print(f"Parquet file saved to {output_file_path}")


def get_device(device_str: Optional[str]) -> torch.device:
    if device_str is not None:
        assert device_str in ['cpu', 'cuda'], f'Incorrect device: {device_str}'
        device = torch.device(device_str)
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return device


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument('--urls-file-path', type=os.path.abspath,
                        help = 'A path to a file utf-8 file with urls. ' \
                                'Each url is in a new line.')
    parser.add_argument('--model-name', type=str, default="sberbank-ai/ruElectra-small",
                        help = 'A name of the hugingface model to use for tokenization. ' \
                                'Default is "ai-forever/ruBert-base"')
    parser.add_argument('--batch-size', type=int, default=10000,
                        help = 'A batch size for the dataloader. Default is 10000')
    parser.add_argument('--num-workers', type=int, default=0,
                        help = 'A number of workers for the dataloader. Default is 0')
    parser.add_argument('--device', type=str, default=None,
                        help = 'A device to use for the model. Default is "cuda" if available, else "cpu"')
    parser.add_argument('--output-file-dir', type=os.path.abspath,
                        help = 'A path to a file where to save the results')
    args = parser.parse_args()
    return args


In [33]:

# if __name__ == "__main__":
 

In [34]:

#     args = parse_args()

args = argparse.Namespace(
    urls_file_path = '/kaggle/input/mts-urls/urls.txt',
    model_name = 'sberbank-ai/ruElectra-small',
    batch_size= 10000,
    num_workers = 0,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    output_file_dir = '/kaggle/working/'
)



In [35]:

device = get_device(args.device)
hf_tokenizer = AutoTokenizer.from_pretrained(args.model_name)
model = AutoModel.from_pretrained(args.model_name)
embedder = MeanPoolSeqEmbedder(model).to(device)


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [36]:
logger.info(f'possible chars: {POSSIBLE_CHARS}')


2024-09-17 21:01:17,341 - INFO - possible chars: {'0', 'm', 'i', '5', 'р', 'g', 'г', '.', 'e', 'х', 'л', 'к', 'ч', 'x', 'н', 't', 'е', 'я', 'c', 'j', 'д', '2', '4', 'п', 'с', 'f', 'ь', 'ц', 'k', 'u', 'и', 'у', 'ъ', 'y', 'в', 'n', 'э', '7', 'z', 'й', 'q', 'v', 'ж', '8', 'w', 'о', 'а', 'ф', 'ш', 'ю', 'ы', '_', 'м', 'h', '1', '3', '6', 'b', '9', 'p', '-', 'т', 'a', 'з', 'r', 'l', 'o', 'б', 's', 'd', 'щ'}


In [37]:
with open(args.urls_file_path, encoding = 'utf-8') as f:
    url_hosts = f.read().splitlines()

chars_met = {c for s in url_hosts for c in s}
# <= operator on sets checks if the left set is a subset of the right set
assert chars_met <= POSSIBLE_CHARS, f'Unexpected characters met: {chars_met - POSSIBLE_CHARS}'

if chars_met == POSSIBLE_CHARS:
    logger.info('All possible characters met, no impossible characters met')



2024-09-17 21:01:17,524 - INFO - All possible characters met, no impossible characters met


In [38]:
one_word_url = [url for url in url_hosts if '.' not in url]

In [39]:
one_word_non_number = [url for url in one_word_url if not set(url) <= set('-0123456789')]

In [40]:
logger.info(f'One word non-number urls: {one_word_non_number}')

2024-09-17 21:01:17,563 - INFO - One word non-number urls: ['localhost', 'unknown', 'serialy', 'chrome-extension', 'file', 'test', 'null', 'undefined', 't', '_', 'home', 'downloads', 'content', 'about', 'false', 'media']


In [41]:
mixed_lang_domains = get_mixed_lang_domains(url_hosts)

logger.info(f'Mixed language domains number: {len(mixed_lang_domains)}')
logger.info(f'Mixed language domains: {mixed_lang_domains}')


2024-09-17 21:01:20,180 - INFO - Mixed language domains number: 158
2024-09-17 21:01:20,181 - INFO - Mixed language domains: {'автоинфо-рус.turbopages', 'димаш-кудайберген-игорь-крутой.vuxo7', 'проконспект-рф.turbopages', 'про--бабло-рф.turbopages', 'влпк-рф.turbopages', 'букмекеры-рф.turbopages', '100--советов-рф.turbopages', 'гражданство-online.turbopages', 'пгс24-рус.turbopages', 'экономщик-рф.turbopages', 'ладухин.vuxo7', 'венер-хурматуллин.vuxo7', 'микориза-рф.turbopages', 'втораяиндустриализация-рф.turbopages', 'технокад.pbprog', 'с-ермолаево.р-н-куюргазинский.webufa', 'аква45-рф.turbopages', 'курс--доллара-онлайн.turbopages', 'г-стерлитамак.webufa', 'с-чесноковка.р-н-уфимский.webufa', 'егрн--документ-рф.turbopages', 'мк--союз-рф.turbopages', 'с-инзер.р-н-белорецкий.webufa', 'с-нагаево.г-уфа.webufa', 'питомник--сад-рф.turbopages', 'маминов-рф.turbopages', 'неизвестен.vuxo7', 'ппни-рф.turbopages', 'счет.h1n', 'мимо.com', 'гутенклик-рф.turbopages', 'эрекция-рф.turbopages', 'с-павло

In [42]:
status_to_urls = defaultdict(list)

for url in tqdm(url_hosts):
    orig_url = url
    status, words = get_words(url, preprocess = True)
    status_to_urls[status].append(orig_url)


status_to_n_urls = {k: len(v) for k, v in status_to_urls.items()}

logger.info(f'Statuses to number of urls: {status_to_n_urls}')

n_urls = 10
for status, urls in status_to_urls.items():
    print(f"{status}:")
    print("-----")
    urls = urls[:n_urls]
    words_lsts = [get_words(url, preprocess = True)[1] for url in urls]
    for url in urls[:n_urls]:
        print(url)
        if is_punycode(url):
            print(f"after puny_code conversion: {convert_punycode(url)}")
        print(get_words(url, preprocess=True)[1])
    print()


100%|██████████| 199683/199683 [00:13<00:00, 14939.38it/s]

2024-09-17 21:01:33,569 - INFO - Statuses to number of urls: {'proper_translit': 173790, 'has_xwq': 18969, 'no_eng_no_ru': 1023, 'initially_ru': 5708, 'transliterated. cyr_and_latin_before_translit proper_translit': 119, 'no_cyrillic_after_translit has_xwq': 35, 'transliterated. cyr_and_latin_before_translit has_xwq': 39}


proper_translit:
-----
torg.1777.ru
['торг']
maykop.kinoafisha.info
['маыкоп', 'киноафиша']
schoolsushi.ru
['счоолсуши']
essentuki.compromesso.ru
['ессентуки', 'цомпромессо']
vasilievaa.narod.ru
['василиеваа', 'народ']
russiapost.su
['руссиапост']
12monet.ru
['монет']
mrttula.ru
['мрттула']
kimry-meb.ru
['кимры', 'меб']
vgstroy.ru
['вгстроы']

has_xwq:
-----
msk.rdw.ru
['мск', 'рд']
webznam.ru
['ебзнам']
taxizagorod.ru
['та', 'изагород']
makhachkala.cataloxy.ru
['макхачкала', 'цатало', 'ы']
news.scourier.ru
['не', 'с', 'сцоуриер']
belgorod.sidex.ru
['белгород', 'сиде']
imwelder.ru
['им', 'елдер']
wwii-soldat.narod.ru
['ии', 'солдат', 'народ']
wotblitz.ru
['отблитз']
opexobo-3yebo.ru
['опе', 'обо', 'ыебо']

no_eng_no_ru:
-----
3652.ru
[]
175
[]
179
[]
38
[]
89057294540.ru
[]
210
[]
85.233.77.7
[]
5.101.123.242
[]
1770.ru
[]
81
[]

initially_ru:
-----
xn--n1aeb2a.xn--p1ai
after puny_code conversion: сшор.рф
['сшор']
xn----7sbaab9degjhhhuo.xn--p1ai
after puny_code conversion: панорама-пла

In [43]:
preprocessed_urls = list(set(preprocess_url(url) for url in url_hosts))

In [44]:
'' in preprocessed_urls

True

In [45]:
len(preprocessed_urls)

196693

In [46]:
preprocessed_urls[:10]

['',
 'uznayonline',
 'mebetal',
 'belohota',
 'toratarot.forum2x2',
 'udmurtskaya-respublika.unibo',
 'nachert',
 'volzhskiy.unibo',
 'arbat-pravo',
 'tihvin.yavitrina']

In [47]:
second_domains = [url.split('.')[-1] for url in preprocessed_urls]


In [48]:

from collections import Counter

secon_domains_freqs = Counter(second_domains)

In [49]:
secon_domains_freqs

Counter({'turbopages': 12835,
         'livejournal': 8092,
         'narod': 3054,
         'ucoz': 2629,
         'bezformata': 1305,
         'hh': 934,
         'academic': 874,
         'com': 865,
         'fandom': 786,
         'mirtesen': 596,
         'jsprav': 594,
         'spravker': 580,
         'gde': 571,
         'superjob': 533,
         'jobfilter': 505,
         'vsn': 472,
         'blizko': 444,
         'nuipogoda': 367,
         'cataloxy': 340,
         'blogspot': 332,
         'at': 326,
         'my1': 320,
         'kinoafisha': 310,
         'pulscen': 278,
         '3dn': 269,
         'build2last': 255,
         'spravka-region': 250,
         'bankiros': 247,
         'unibo': 238,
         'timeweb': 233,
         'bizly': 219,
         'kitabi': 209,
         '1000bankov': 206,
         'alloy': 205,
         'clan': 202,
         'forum2x2': 197,
         'barahla': 196,
         'yavitrina': 195,
         'svetoforonline': 194,
         'mbib': 191

In [50]:
url_tokenizer = UrlTokenizer(hf_tokenizer, preprocessing_needed = False)

example_url = 'privet.ml.inzhiner.com'
example_url_tokenized = url_tokenizer(example_url)
logger.info(f'Example url: {example_url}')
logger.info(f'Example url tokenized: {example_url_tokenized}')
logger.info(f'Example url tokenized shape: {example_url_tokenized["input_ids"].shape}')
logger.info(f'Example url decoded: {url_tokenizer.tokenizer.decode(example_url_tokenized["input_ids"][0])}')

2024-09-17 21:01:36,242 - INFO - Example url: privet.ml.inzhiner.com
2024-09-17 21:01:36,246 - INFO - Example url tokenized: {'input_ids': tensor([[     2, 190125, 146858, 124529,  20255, 240359,  22290,      3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
2024-09-17 21:01:36,247 - INFO - Example url tokenized shape: torch.Size([1, 8])
2024-09-17 21:01:36,248 - INFO - Example url decoded: [CLS] привет мл инжинер цом [SEP]


In [51]:
url_tokenizer = UrlTokenizer(hf_tokenizer, preprocessing_needed = False)

example_url = ''
example_url_tokenized = url_tokenizer(example_url)
logger.info(f'Example url: {example_url}')
logger.info(f'Example url tokenized: {example_url_tokenized}')
logger.info(f'Example url tokenized shape: {example_url_tokenized["input_ids"].shape}')
logger.info(f'Example url decoded: {url_tokenizer.tokenizer.decode(example_url_tokenized["input_ids"][0])}')

2024-09-17 21:01:36,258 - INFO - Example url: 
2024-09-17 21:01:36,259 - INFO - Example url tokenized: {'input_ids': tensor([[2, 3]]), 'token_type_ids': tensor([[0, 0]]), 'attention_mask': tensor([[1, 1]])}
2024-09-17 21:01:36,260 - INFO - Example url tokenized shape: torch.Size([1, 2])
2024-09-17 21:01:36,261 - INFO - Example url decoded: [CLS] [SEP]


In [53]:
url_dataset = UrlDataset(preprocessed_urls, hf_tokenizer, preprocessing_needed = False)

url_dataloader = DataLoader(url_dataset, batch_size=args.batch_size, 
                            shuffle=False, collate_fn = collate_fn,
                            num_workers=args.num_workers)


all_outputs = []

with torch.inference_mode():
    for input_ids, attention_masks in tqdm(url_dataloader):
        input_ids=input_ids.to(args.device)
        attention_masks=attention_masks.to(args.device)
        sentence_embeddings = embedder(tokenized = input_ids, attention_mask = attention_masks) 
        all_outputs.append(sentence_embeddings.to('cpu'))

all_embeddings = torch.cat(all_outputs, dim = 0)

all_embeddings = all_embeddings.numpy()

assert len(all_embeddings) == len(preprocessed_urls)

In [67]:

embeddings_list = list(all_embeddings)


data = {'url_host_preprocessed': preprocessed_urls, 'url_host_embedding': embeddings_list}

df = pd.DataFrame(data)

In [68]:
logger.info('\n' + str(df.head()))

2024-09-17 21:28:34,403 - INFO - 
  url_host_preprocessed                                 url_host_embedding
0                        [0.01859397, 0.28440517, 0.064282626, -0.02738...
1           uznayonline  [0.018607048, 0.3694605, 0.059723165, -0.02284...
2               mebetal  [0.018624451, 0.21577111, 0.05554186, -0.02540...
3              belohota  [0.01864303, 0.22606, 0.049342472, -0.02587692...
4    toratarot.forum2x2  [0.0186494, 0.01898303, 0.047724646, -0.023608...


In [69]:
df.head()

,url_host_preprocessed,url_host_embedding
0,,"[0.01859397, 0.28440517, 0.064282626, -0.02738..."
1,uznayonline,"[0.018607048, 0.3694605, 0.059723165, -0.02284..."
2,mebetal,"[0.018624451, 0.21577111, 0.05554186, -0.02540..."
3,belohota,"[0.01864303, 0.22606, 0.049342472, -0.02587692..."
4,toratarot.forum2x2,"[0.0186494, 0.01898303, 0.047724646, -0.023608..."


In [70]:
out_fname = get_output_file_name(input_file_path = args.urls_file_path, model_name = args.model_name, extension = 'parquet')
output_path = os.path.join(args.output_file_dir, out_fname)
df.to_parquet(output_path, engine='pyarrow', index=False)

print(f"Parquet file saved to {output_path}")

Parquet file saved to /kaggle/working/urls.txt_sberbank-ai_ruElectra-small.parquet


In [ ]:
get_emb = lambda url: df[df['url_host'] == url]['url_host_embedding'].values[0]

In [ ]:
torch.equal(get_emb('1'), get_emb('4-x-4.ru'))

In [ ]:
torch.equal(get_emb('1'), get_emb('yandex.ru'))

In [ ]:
emty_seq_tokenized  = hf_tokenizer('', return_tensors="pt")
empty_seq_emb = embedder(emty_seq_tokenized['input_ids'].to(device), emty_seq_tokenized['attention_mask'].to(device))

In [ ]:
empty_seq_emb.shape

In [ ]:
torch.equal(get_emb('1'), empty_seq_emb.cpu())

In [ ]:
empty_seq_emb.shape

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-10)
    return sum_embeddings / sum_mask
#Sentences we want sentence embeddings for
sentences = ['Привет! Как твои дела?',
             'А правда, что 42 твое любимое число?']
#Load AutoModel from huggingface model repository
tokenizer = AutoTokenizer.from_pretrained("sberbank-ai/ruElectra-medium")
model = AutoModel.from_pretrained("sberbank-ai/ruElectra-medium")
#Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, max_length=24, return_tensors='pt')
#Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)
#Perform pooling. In this case, mean pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])


In [ ]:
sentence_embeddings

In [ ]:
model_output[0].shape

In [ ]:
model_output[0]

In [ ]:
encoded_input['attention_mask']

In [ ]:
torch.equal(model_output['last_hidden_state'], model_output[0]) 